# MM-Net — permutation importance, and whether it agrees with the ablation

The paper's interpretability claim rests on a leave-one-out ablation: remove a modality,
retrain, and see which head suffers. That is a claim about what the model *needs*.

Permutation importance asks a different question about the *same* trained model: shuffle
one modality's features across epochs, so the channel is still present but carries no
information aligned to the epoch, and measure the drop. That is a claim about what the
model *uses*.

The two can disagree. A model can need a channel during training (because it shapes the
learned representation) yet barely weight it at inference, or lean on a channel that a
retrained model would have compensated for. **Agreement between them is evidence for the
attribution claim that neither method provides alone** — and it is falsifiable, which the
t-SNE panel is not.

No retraining is needed per modality here: one set of ten fold models is trained, then
each modality is permuted at inference.

In [ ]:
import json
import os
import sys
import time

import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..", ".."))
sys.path.insert(0, os.path.join(REPO, "MMNet_research", "model"))
import mmnet_core as C  # noqa: E402

OUT = os.path.join(REPO, "MMNet_research", "results", "revision", "runs")
os.makedirs(OUT, exist_ok=True)

# modality -> (which stream, column selector)
GROUPS = {
    "EEG":       ("eeg", C.EEGM["eeg"]),
    "EOG":       ("eeg", C.EEGM["eog"]),
    "EMG":       ("eeg", C.EEGM["emg"]),
    "SpO2":      ("card", C.CARD_GROUPS["spo2"]),
    "pulse/HRV": ("card", C.CARD_GROUPS["pulse_hrv"]),
    "ECG":       ("card", C.CARD_GROUPS["ecg"]),
    "airflow":   ("card", C.CARD_GROUPS["airflow"]),
    "effort":    ("card", C.CARD_GROUPS["effort"]),
}
print("device:", C.DEV, "| modalities:", list(GROUPS))

## 1. Train the ten folds and permute each modality at inference

Permutation is applied per patient, shuffling the rows of that modality's columns. This
destroys the epoch-to-epoch alignment while preserving each feature's marginal
distribution, so the drop is attributable to information rather than to a distribution
shift. Each permutation is repeated with three different shuffles and averaged.

In [ ]:
N_REPEAT = 3
t0 = time.time()
fold_rows = []

for fi, (tr_all, te) in enumerate(C.FOLDS):
    rng = np.random.RandomState(100 + fi)
    tr_all = list(tr_all); rng.shuffle(tr_all)
    nv = max(10, len(tr_all) // 9)
    va, tr = tr_all[:nv], tr_all[nv:]
    model = C.train_fold(tr, va, "concat", [], [], seed=42)

    # baseline for this fold, no permutation
    ys, ps, ay, ap = [], [], [], []
    for s in te:
        fe, fc, y, a = C.DATA[s]
        sp, apn = C.infer_arrays(model, fe, fc, len(y))
        ys.append(y); ps.append(sp.argmax(1)); ay.append(a); ap.append(apn)
    base_acc = accuracy_score(np.concatenate(ys), np.concatenate(ps))
    base_auc = roc_auc_score(np.concatenate(ay), np.concatenate(ap))

    row = {"fold": fi, "base_acc": base_acc, "base_auc": base_auc}
    for gname, (stream, cols) in GROUPS.items():
        accs, aucs = [], []
        for rep in range(N_REPEAT):
            r = np.random.RandomState(1000 * fi + rep)
            ys, ps, ay, ap = [], [], [], []
            for s in te:
                fe, fc, y, a = C.DATA[s]
                fe2, fc2 = fe.copy(), fc.copy()
                perm = r.permutation(len(y))
                if stream == "eeg":
                    fe2[:, cols] = fe2[perm][:, cols]
                else:
                    fc2[:, cols] = fc2[perm][:, cols]
                sp, apn = C.infer_arrays(model, fe2, fc2, len(y))
                ys.append(y); ps.append(sp.argmax(1)); ay.append(a); ap.append(apn)
            accs.append(accuracy_score(np.concatenate(ys), np.concatenate(ps)))
            aucs.append(roc_auc_score(np.concatenate(ay), np.concatenate(ap)))
        row[gname] = {"acc": float(np.mean(accs)), "auc": float(np.mean(aucs))}
    fold_rows.append(row)
    print("fold %d done (%.1f min elapsed)" % (fi, (time.time() - t0) / 60))

print("\ntotal %.1f min" % ((time.time() - t0) / 60))

## 2. Importance = drop from the unpermuted baseline

Positive numbers mean performance fell when that modality was scrambled.

In [ ]:
base_acc = np.mean([r["base_acc"] for r in fold_rows])
base_auc = np.mean([r["base_auc"] for r in fold_rows])
print("unpermuted baseline: staging acc %.4f | respiratory AUC %.4f\n" % (base_acc, base_auc))

print("%-11s %22s %22s" % ("", "staging acc drop", "respiratory AUC drop"))
print("%-11s %11s %10s %11s %10s" % ("modality", "mean", "sd", "mean", "sd"))
print("-" * 57)

imp = {}
for g in GROUPS:
    da = [r["base_acc"] - r[g]["acc"] for r in fold_rows]
    du = [r["base_auc"] - r[g]["auc"] for r in fold_rows]
    imp[g] = {"acc_drop": float(np.mean(da)), "acc_sd": float(np.std(da)),
              "auc_drop": float(np.mean(du)), "auc_sd": float(np.std(du))}
    print("%-11s %11.4f %10.4f %11.4f %10.4f"
          % (g, np.mean(da), np.std(da), np.mean(du), np.std(du)))

## 3. Does permutation agree with the retraining ablation?

The leave-one-out ablation in the paper retrains without each modality. If the two
methods rank the modalities the same way, the attribution is supported by two independent
lines of evidence rather than one.

In [ ]:
from scipy.stats import spearmanr

# published leave-one-out drops (full - ablated), Table VI
ABLATION = {
    "SpO2":      {"acc": 0.723 - 0.727, "auc": 0.711 - 0.681},
    "effort":    {"acc": 0.723 - 0.728, "auc": 0.711 - 0.726},
    "pulse/HRV": {"acc": 0.723 - 0.723, "auc": 0.711 - 0.700},
    "ECG":       {"acc": 0.723 - 0.723, "auc": 0.711 - 0.711},
    "airflow":   {"acc": 0.723 - 0.722, "auc": 0.711 - 0.704},
    "EOG":       {"acc": 0.723 - 0.712, "auc": 0.711 - 0.705},
    "EMG":       {"acc": 0.723 - 0.724, "auc": 0.711 - 0.707},
}
shared = [g for g in ABLATION if g in imp]

print("%-11s %12s %12s %12s %12s" % ("modality", "perm dAUC", "abl dAUC", "perm dAcc", "abl dAcc"))
print("-" * 62)
for g in sorted(shared, key=lambda x: -imp[x]["auc_drop"]):
    print("%-11s %12.4f %12.4f %12.4f %12.4f"
          % (g, imp[g]["auc_drop"], ABLATION[g]["auc"],
             imp[g]["acc_drop"], ABLATION[g]["acc"]))

for metric, key in (("respiratory AUC", "auc"), ("staging accuracy", "acc")):
    a = [imp[g][key + "_drop"] for g in shared]
    b = [ABLATION[g][key] for g in shared]
    rho, p = spearmanr(a, b)
    print("\n%s: Spearman rho = %.3f (p = %.3f, n = %d)" % (metric, rho, p, len(shared)))

In [ ]:
json.dump({"baseline": {"acc": float(base_acc), "auc": float(base_auc)},
           "importance": imp, "per_fold": fold_rows},
          open(os.path.join(OUT, "permutation_importance.json"), "w"), indent=1)
print("wrote permutation_importance.json")

## Reading the result

The prediction the paper's attribution claim makes is specific: scrambling the
cardiorespiratory channels should cost respiratory AUC and leave staging alone, and
scrambling the ocular channels should do the reverse. Both are visible in the table above
or they are not — and unlike a t-SNE, this could have come out otherwise.